In [ ]:
import sys
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[1] if NOTEBOOK_DIR.name == "data_exploration" else NOTEBOOK_DIR
SRC_ROOT = REPO_ROOT / "src"
DATA_ROOT = REPO_ROOT / "data"

if str(SRC_ROOT) not in sys.path:
    sys.path.append(str(SRC_ROOT))

from data_exploration.data_exploration import (
    DATA_ROOT as MODULE_DATA_ROOT,
)
from data_exploration.data_exploration import (
    REPO_ROOT as MODULE_REPO_ROOT,
)
from data_exploration.data_exploration import (
    candidate_verdict_table,
    dataset_registry,
    get_dataset_config,
    inspect_dataset_file,
    schema_overview,
    summarize_dataset,
    to_json_ready,
)
from data_exploration.data_information import (
    build_markdown_report,
    resolve_paths,
    scan_dataset,
)

assert REPO_ROOT == MODULE_REPO_ROOT
assert DATA_ROOT == MODULE_DATA_ROOT

In [ ]:
registry = dataset_registry()
records = [scan_dataset(DATA_ROOT / dataset_name) for dataset_name in registry["dataset"]]
inventory_rows = [
    {
        "dataset": record.name,
        "size_mib": round(record.total_bytes / (1024**2), 2),
        "files": len(record.file_records),
        "status": record.status,
        "pairwise_ready": (record.exploration_summary or {})
        .get("edgrec_requirements", {})
        .get("supports_pairwise_triplets"),
        "sign_ready": (record.exploration_summary or {})
        .get("edgrec_requirements", {})
        .get("supports_sign_split"),
        "preprocessing_cost": (record.exploration_summary or {})
        .get("edgrec_requirements", {})
        .get("preprocessing_cost"),
    }
    for record in records
]

display(pd.DataFrame(inventory_rows).sort_values("size_mib", ascending=False))

/home/lazar/Documents/MSc_Data_Science/MSc_Thesis/Efficient-Disentangled-Graph-Recommender/src/data_exploration
['interventions', 'evaluation', 'baselines', 'profiling', 'data_exploration', 'utils']
['uv.lock', 'src', 'main.py', '.vscode', 'docs', '.python-version', 'LICENCE', 'latex', 'README.md', '.gitignore', '.github', 'data', 'scripts', '.venv', 'pyproject.toml', '.git', 'external', 'results', 'experiments']


In [ ]:
selected_dataset = "MovieLens1M"
selected_summary = summarize_dataset(selected_dataset)
selected_record = scan_dataset(DATA_ROOT / selected_dataset)

selected_overview = {
    "dataset": selected_dataset,
    "kind": selected_summary["kind"],
    "root": selected_summary["root"],
    "present_files": selected_summary["present_files"],
    "missing_files": selected_summary["missing_files"],
    **selected_summary["edgrec_requirements"],
    "note": selected_summary["note"],
}

selected_overview

--- ../../data/MovieLens1M/raw/movies.dat ---
1::Toy Story (1995)::Animation|Children's|Comedy
2::Jumanji (1995)::Adventure|Children's|Fantasy
3::Grumpier Old Men (1995)::Comedy|Romance
4::Waiting to Exhale (1995)::Comedy|Drama
5::Father of the Bride Part II (1995)::Comedy
6::Heat (1995)::Action|Crime|Thriller
7::Sabrina (1995)::Comedy|Romance
8::Tom and Huck (1995)::Adventure|Children's
9::Sudden Death (1995)::Action
10::GoldenEye (1995)::Action|Adventure|Thriller
--- ../../data/MovieLens1M/raw/ratings.dat ---
1::1193::5::978300760
1::661::3::978302109
1::914::3::978301968
1::3408::4::978300275
1::2355::5::978824291
1::1197::3::978302268
1::1287::5::978302039
1::2804::5::978300719
1::594::4::978302268
1::919::4::978301368
--- ../../data/MovieLens1M/raw/users.dat ---
1::F::1::10::48067
2::M::56::16::70072
3::M::25::15::55117
4::M::45::7::02460
5::M::25::20::55455
6::F::50::9::55117
7::M::35::1::06810
8::M::25::12::11413
9::M::25::17::61614
10::F::35::1::95370


In [ ]:
file_inventory_rows = []
for record in sorted(
    selected_record.file_records,
    key=lambda item: (-item.size_bytes, item.relative_path),
):
    schema = record.schema or {}
    file_inventory_rows.append(
        {
            "relative_path": record.relative_path,
            "size_mib": round(record.size_bytes / (1024**2), 2),
            "line_count": record.line_count,
            "parsed_count": record.parsed_count,
            "column_count": schema.get("column_count"),
            "columns": ", ".join(schema.get("columns", [])[:12]),
            "details": record.details,
        },
    )

display(pd.DataFrame(file_inventory_rows))

In [ ]:
selected_file = get_dataset_config(selected_dataset)["files"][0]
selected_file_path = Path(get_dataset_config(selected_dataset)["root"]) / selected_file
selected_file_inspection = inspect_dataset_file(selected_dataset, selected_file_path)

file_schema = schema_overview(selected_file_inspection)
file_details = {
    "dataset": selected_dataset,
    "file": selected_file,
    "path": selected_file_inspection["path"],
    "inspection_type": selected_file_inspection.get("type"),
    "column_count": file_schema.get("column_count"),
    "columns": file_schema.get("columns"),
    "delimiter": file_schema.get("delimiter"),
    "dtypes": selected_file_inspection.get("dtypes"),
    "sample_rows": selected_file_inspection.get("sample_rows"),
    "variables": selected_file_inspection.get("variables"),
    "keys": selected_file_inspection.get("keys"),
    "shape": selected_file_inspection.get("shape"),
}

file_details

In [ ]:
comparison_datasets = [
    "MovieLens1M",
    "Taobao",
    "KuaiRec_v2",
    "KuaiRand-1K",
    "AmazonBook",
]
comparison_table = candidate_verdict_table(comparison_datasets).sort_values(
    ["preprocessing_cost", "dataset"],
    ascending=[True, True],
)

display(comparison_table)

movie_id,title,genres
u16,str,str
1,"""Toy Story (1995)""","""Animation|Children's|Comedy"""
2,"""Jumanji (1995)""","""Adventure|Children's|Fantasy"""
3,"""Grumpier Old Men (1995)""","""Comedy|Romance"""
4,"""Waiting to Exhale (1995)""","""Comedy|Drama"""
5,"""Father of the Bride Part II (1…","""Comedy"""
…,…,…
3948,"""Meet the Parents (2000)""","""Comedy"""
3949,"""Requiem for a Dream (2000)""","""Drama"""
3950,"""Tigerland (2000)""","""Drama"""


user_id,movie_id,rating,timestamp
u16,u16,u8,datetime[ms]
1,1193,5,1970-01-12 07:45:00.760
1,661,3,1970-01-12 07:45:02.109
1,914,3,1970-01-12 07:45:01.968
1,3408,4,1970-01-12 07:45:00.275
1,2355,5,1970-01-12 07:53:44.291
…,…,…,…
6040,1091,1,1970-01-12 01:45:16.541
6040,1094,5,1970-01-12 01:45:04.887
6040,562,5,1970-01-12 01:45:04.746


user_id,gender,age,occupation,zip_code
u16,str,u8,u8,str
1,"""F""",1,10,"""48067"""
2,"""M""",56,16,"""70072"""
3,"""M""",25,15,"""55117"""
4,"""M""",45,7,"""02460"""
5,"""M""",25,20,"""55455"""
…,…,…,…,…
6036,"""F""",25,15,"""32603"""
6037,"""F""",45,1,"""76006"""
6038,"""F""",56,1,"""14706"""


In [ ]:
data_root, report_path = resolve_paths(None)
report = build_markdown_report(records, data_root)
report_path.write_text(report, encoding="utf-8")

print(report_path)
print(report.splitlines()[:40])

In [ ]:
print(to_json_ready(selected_summary))

['indices', 'indptr', 'format', 'shape', 'data']


column_0,column_1,column_2,column_3,column_4,column_5,column_6,column_7,column_8,column_9,column_10,column_11,column_12,column_13,column_14,column_15,column_16,column_17,column_18,column_19,column_20,column_21,column_22,column_23,column_24,column_25,column_26,column_27,column_28,column_29,column_30,column_31,column_32,column_33,column_34,column_35,column_36,…,column_163,column_164,column_165,column_166,column_167,column_168,column_169,column_170,column_171,column_172,column_173,column_174,column_175,column_176,column_177,column_178,column_179,column_180,column_181,column_182,column_183,column_184,column_185,column_186,column_187,column_188,column_189,column_190,column_191,column_192,column_193,column_194,column_195,column_196,column_197,column_198,column_199
f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
-0.146565,0.222557,-0.359671,0.150984,-0.071912,-0.235524,-0.399557,0.248068,-0.304489,0.645639,-0.130156,-0.013443,0.354324,0.014126,-0.101768,0.014744,0.630983,0.000273,0.096278,0.010879,-0.307503,-0.496155,-0.112764,-0.097032,0.015852,0.298255,0.016043,0.33084,0.239766,0.435762,-0.239793,0.217937,0.068402,0.073455,-0.050774,-0.435119,0.043131,…,0.614012,-0.424537,-0.164985,-0.482328,-0.004774,-0.014464,-0.215895,0.047029,-0.047218,0.087692,0.057601,-0.070737,-0.033259,0.166638,0.125321,-0.088582,0.011125,-0.171835,0.109636,0.078539,-0.14029,-1.447082,0.034595,1.539603,-0.296689,-1.022162,1.253607,-0.253967,-0.786987,-1.98066,-1.165259,0.19669,-0.245297,-0.163149,0.169936,0.89743,1.65267
-0.28046,0.019013,0.430075,0.51524,0.330475,-0.419859,-0.544838,0.529882,-0.801702,-0.237201,0.054786,0.326187,-0.966576,0.868007,-0.432585,0.488616,0.010137,0.130631,-0.703846,-1.300433,-0.317771,-0.422018,0.35965,0.042782,-0.948256,-0.323537,0.902857,-0.365816,-0.204076,-0.389284,-0.278618,-0.917009,-0.175746,0.249574,0.643533,-0.013158,0.212785,…,0.103452,-0.225345,-0.13175,-0.51851,0.353996,0.441673,0.316056,0.343886,-0.085074,0.384994,0.931415,-0.153713,-0.248714,-0.036018,-0.30593,0.001372,0.283696,-0.196725,0.398403,-0.207962,0.107183,-0.544531,0.109079,0.450553,1.329944,0.914631,-0.150199,0.220331,0.210623,-0.514243,-0.885525,-0.722589,-1.649528,0.345465,-1.175781,-1.836544,-1.169346
0.255406,0.251893,-0.029144,-0.401508,-0.347818,-0.215458,0.173424,0.008264,-0.223711,0.112321,-0.326059,-0.26283,-0.303891,-0.186627,-0.395915,-0.115166,-0.096325,-0.311859,-0.007535,0.037566,0.061081,-0.018451,-0.149268,-0.149565,0.131302,-0.019817,0.11653,0.154304,0.200508,0.314093,0.078572,0.017807,0.192564,0.175227,-0.289983,-0.051777,-0.060609,…,-0.186748,0.055057,-0.103086,-0.201133,-0.062955,0.266317,-0.228652,-0.236443,-0.042799,-0.096797,-0.067112,0.225485,0.148724,0.157277,-0.088592,-0.126146,-0.05615,-0.034107,-0.040069,0.134773,-0.171421,0.628028,0.389593,-0.414096,-1.608646,-0.364955,0.333451,-0.066612,-1.044846,0.727054,0.037794,-1.630903,0.66492,-0.973265,1.375145,-0.073495,0.626154
-0.211678,0.409369,-0.080767,0.008406,0.125527,-0.014299,-0.482628,-0.781813,0.305528,-0.147415,0.42539,-0.457458,0.189899,0.23573,0.333011,-0.245437,0.084169,-0.294255,0.259022,0.173178,0.245103,0.099009,-0.224773,-0.382108,0.043738,-0.530452,-0.161229,-0.150242,0.280151,0.090318,-0.163988,0.075461,-0.209615,-0.312215,0.127102,-0.513184,0.037773,…,-0.031811,0.220412,0.06178,-0.046205,0.15344,0.10734,-0.082938,-0.126164,0.188587,0.200689,0.212696,0.017781,-0.414163,0.09253,0.44286,0.031408,-0.051742,-0.110185,0.06832,0.254647,-0.037876,-0.464468,-0.937275,0.145995,-0.184261,-0.766594,-0.225597,-0.659484,1.378926,1.328842,0.689907,-0.563646,-0.090562,1.55757,-1.46703,-0.185993,-0.246122
0.110547,0.175176,-0.219198,-0.115018,-0.175498,0.226963,0.022288,-0.109078,-0.234378,-0.350186,-0.022846,-0.304968,-0.214338,-0.1